In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# 1. 准备数据（自动下载 MNIST 数据集，大概几十MB）
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# 2. 制作“任务 1”的数据：只包含数字 0 和 1
idx_task1_train = [i for i, label in enumerate(train_dataset.targets) if label in [0, 1]]
train_loader_task1 = DataLoader(Subset(train_dataset, idx_task1_train), batch_size=64, shuffle=True)
idx_task1_test = [i for i, label in enumerate(test_dataset.targets) if label in [0, 1]]
test_loader_task1 = DataLoader(Subset(test_dataset, idx_task1_test), batch_size=64, shuffle=False)

# 3. 制作“任务 2”的数据：只包含数字 2 和 3
idx_task2_train = [i for i, label in enumerate(train_dataset.targets) if label in [2, 3]]
train_loader_task2 = DataLoader(Subset(train_dataset, idx_task2_train), batch_size=64, shuffle=True)
idx_task2_test = [i for i, label in enumerate(test_dataset.targets) if label in [2, 3]]
test_loader_task2 = DataLoader(Subset(test_dataset, idx_task2_test), batch_size=64, shuffle=False)

print("数据准备完毕！")

# 定义一个非常简单的神经网络
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 4) # 为什么是4？因为我们总共只做0,1,2,3这4个数字的分类
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 训练函数
def train(model, loader, optimizer, criterion, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in loader:
            # 注意：这里的 target 需要映射一下，因为原始标签是0,1,2,3，我们的网络输出是0,1,2,3
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

# 测试准确率的函数
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(loader.dataset)

print("--- 阶段 1：学习任务 1（数字 0 和 1） ---")
train(model, train_loader_task1, optimizer, criterion, epochs=2)
acc1_after_task1 = evaluate(model, test_loader_task1)
print(f"学完任务1，模型对任务1的准确率: {acc1_after_task1:.2f}%\n")

print("--- 阶段 2：直接学习任务 2（数字 2 和 3） ---")
train(model, train_loader_task2, optimizer, criterion, epochs=2)
acc2_after_task2 = evaluate(model, test_loader_task2)
print(f"学完任务2，模型对任务2的准确率: {acc2_after_task2:.2f}%")

# 重点来了：看看它还记不记得旧知识
acc1_after_task2 = evaluate(model, test_loader_task1)
print(f"学完任务2后，回头测试任务1的准确率: {acc1_after_task2:.2f}%  <--- 灾难性遗忘发生了！")

# 重新初始化模型
model_replay = SimpleNN()
optimizer_replay = optim.Adam(model_replay.parameters(), lr=0.001)

# 创建一个“记忆库”（保留一部分旧数据）
memory_loader = DataLoader(Subset(train_dataset, idx_task1_train[:1000]), batch_size=32, shuffle=True) # 只记住 1000 张旧图

# 1. 正常学任务 1
print("--- Replay 法：学习任务 1 ---")
train(model_replay, train_loader_task1, optimizer_replay, criterion, epochs=2)

# 学完任务1后
train(model_replay, train_loader_task1, optimizer_replay, criterion, epochs=2)
print("学完任务1后，任务2准确率:", evaluate(model_replay, test_loader_task2))

for epoch in range(2):
    train(model_replay, train_loader_task2, optimizer_replay, criterion, epochs=1)
    print(f"第{epoch + 1}轮学完任务2后:", evaluate(model_replay, test_loader_task2))

    train(model_replay, memory_loader, optimizer_replay, criterion, epochs=1)
    print(f"第{epoch + 1}轮Replay后:", evaluate(model_replay, test_loader_task2))

# 2. 学任务 2 的时候，穿插着复习记忆库里的旧数据
print("--- Replay 法：学习任务 2（边学边复习） ---")
for epoch in range(2):
    # 先学一点任务 2
    train(model_replay, train_loader_task2, optimizer_replay, criterion, epochs=1)
    # 再复习一点任务 1
    train(model_replay, memory_loader, optimizer_replay, criterion, epochs=1)

# 测试最终结果
acc1_replay = evaluate(model_replay, test_loader_task1)
acc2_replay = evaluate(model_replay, test_loader_task2)
print(f"使用 Replay 算法后，任务1准确率: {acc1_replay:.2f}%")
print(f"使用 Replay 算法后，任务2准确率: {acc2_replay:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# 1. 准备数据（自动下载 MNIST 数据集，大概几十MB）
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# 2. 制作“任务 1”的数据：只包含数字 0 和 1
idx_task1_train = [i for i, label in enumerate(train_dataset.targets) if label in [0, 1]]
train_loader_task1 = DataLoader(Subset(train_dataset, idx_task1_train), batch_size=64, shuffle=True)
idx_task1_test = [i for i, label in enumerate(test_dataset.targets) if label in [0, 1]]
test_loader_task1 = DataLoader(Subset(test_dataset, idx_task1_test), batch_size=64, shuffle=False)

# 3. 制作“任务 2”的数据：只包含数字 2 和 3
idx_task2_train = [i for i, label in enumerate(train_dataset.targets) if label in [2, 3]]
train_loader_task2 = DataLoader(Subset(train_dataset, idx_task2_train), batch_size=64, shuffle=True)
idx_task2_test = [i for i, label in enumerate(test_dataset.targets) if label in [2, 3]]
test_loader_task2 = DataLoader(Subset(test_dataset, idx_task2_test), batch_size=64, shuffle=False)

print("数据准备完毕！")

# 定义一个非常简单的神经网络
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 4) # 为什么是4？因为我们总共只做0,1,2,3这4个数字的分类
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 训练函数
def train(model, loader, optimizer, criterion, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in loader:
            # 注意：这里的 target 需要映射一下，因为原始标签是0,1,2,3，我们的网络输出是0,1,2,3
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

# 测试准确率的函数
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(loader.dataset)

print("--- 阶段 1：学习任务 1（数字 0 和 1） ---")
train(model, train_loader_task1, optimizer, criterion, epochs=2)
acc1_after_task1 = evaluate(model, test_loader_task1)
print(f"学完任务1，模型对任务1的准确率: {acc1_after_task1:.2f}%\n")

print("--- 阶段 2：直接学习任务 2（数字 2 和 3） ---")
train(model, train_loader_task2, optimizer, criterion, epochs=2)
acc2_after_task2 = evaluate(model, test_loader_task2)
print(f"学完任务2，模型对任务2的准确率: {acc2_after_task2:.2f}%")

# 重点来了：看看它还记不记得旧知识
acc1_after_task2 = evaluate(model, test_loader_task1)
print(f"学完任务2后，回头测试任务1的准确率: {acc1_after_task2:.2f}%  <--- 灾难性遗忘发生了！")

# 重新初始化模型
model_replay = SimpleNN()
optimizer_replay = optim.Adam(model_replay.parameters(), lr=0.001)

# 创建一个“记忆库”（保留一部分旧数据）
memory_loader = DataLoader(Subset(train_dataset, idx_task1_train[:1000]), batch_size=32, shuffle=True) # 只记住 1000 张旧图

# 1. 正常学任务 1
print("--- Replay 法：学习任务 1 ---")
train(model_replay, train_loader_task1, optimizer_replay, criterion, epochs=2)



# 2. 学任务 2 的时候，穿插着复习记忆库里的旧数据
print("--- Replay 法：学习任务 2（边学边复习） ---")
for epoch in range(2):
    # 先学一点任务 2
    train(model_replay, train_loader_task2, optimizer_replay, criterion, epochs=1)
    # 再复习一点任务 1
    train(model_replay, memory_loader, optimizer_replay, criterion, epochs=1)

# 测试最终结果
acc1_replay = evaluate(model_replay, test_loader_task1)
acc2_replay = evaluate(model_replay, test_loader_task2)
print(f"使用 Replay 算法后，任务1准确率: {acc1_replay:.2f}%")
print(f"使用 Replay 算法后，任务2准确率: {acc2_replay:.2f}%")

model_replay.eval()
preds = []

with torch.no_grad():
    for data, _ in test_loader_task2:
        output = model_replay(data)
        preds.extend(output.argmax(dim=1).tolist())

print("任务2测试集预测分布:", torch.bincount(torch.tensor(preds), minlength=4))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt



# 2. 制作“任务 1”的数据：只包含数字 0 和 1
idx_task1_train = [i for i, label in enumerate(train_dataset.targets) if label in [0, 1]]
train_loader_task1 = DataLoader(Subset(train_dataset, idx_task1_train), batch_size=64, shuffle=True)
idx_task1_test = [i for i, label in enumerate(test_dataset.targets) if label in [0, 1]]
test_loader_task1 = DataLoader(Subset(test_dataset, idx_task1_test), batch_size=64, shuffle=False)

# 3. 制作“任务 2”的数据：只包含数字 2 和 3
idx_task2_train = [i for i, label in enumerate(train_dataset.targets) if label in [2, 3]]
train_loader_task2 = DataLoader(Subset(train_dataset, idx_task2_train), batch_size=64, shuffle=True)
idx_task2_test = [i for i, label in enumerate(test_dataset.targets) if label in [2, 3]]
test_loader_task2 = DataLoader(Subset(test_dataset, idx_task2_test), batch_size=64, shuffle=False)

print("数据准备完毕！")

# 定义一个非常简单的神经网络
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 4) # 为什么是4？因为我们总共只做0,1,2,3这4个数字的分类
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 训练函数
def train(model, loader, optimizer, criterion, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in loader:
            # 注意：这里的 target 需要映射一下，因为原始标签是0,1,2,3，我们的网络输出是0,1,2,3
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

# 测试准确率的函数
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(loader.dataset)

print("--- 阶段 1：学习任务 1（数字 0 和 1） ---")
train(model, train_loader_task1, optimizer, criterion, epochs=2)
acc1_after_task1 = evaluate(model, test_loader_task1)
print(f"学完任务1，模型对任务1的准确率: {acc1_after_task1:.2f}%\n")

print("--- 阶段 2：直接学习任务 2（数字 2 和 3） ---")
train(model, train_loader_task2, optimizer, criterion, epochs=2)
acc2_after_task2 = evaluate(model, test_loader_task2)
print(f"学完任务2，模型对任务2的准确率: {acc2_after_task2:.2f}%")

# 重点来了：看看它还记不记得旧知识
acc1_after_task2 = evaluate(model, test_loader_task1)
print(f"学完任务2后，回头测试任务1的准确率: {acc1_after_task2:.2f}%  <--- 灾难性遗忘发生了！")

# 重新初始化模型
model_replay = SimpleNN()
optimizer_replay = optim.Adam(model_replay.parameters(), lr=0.001)

# 创建一个“记忆库”（保留一部分旧数据）
memory_loader = DataLoader(Subset(train_dataset, idx_task1_train[:1000]), batch_size=32, shuffle=True) # 只记住 1000 张旧图

# 1. 正常学任务 1
print("--- Replay 法：学习任务 1 ---")
train(model_replay, train_loader_task1, optimizer_replay, criterion, epochs=2)



# 2. 学任务 2 的时候，穿插着复习记忆库里的旧数据
print("--- Replay 法：学习任务 2（边学边复习） ---")
for epoch in range(2):
    # 先学一点任务 2
    train(model_replay, train_loader_task2, optimizer_replay, criterion, epochs=1)
    # 再复习一点任务 1
    train(model_replay, memory_loader, optimizer_replay, criterion, epochs=1)

# 测试最终结果
acc1_replay = evaluate(model_replay, test_loader_task1)
acc2_replay = evaluate(model_replay, test_loader_task2)
print(f"使用 Replay 算法后，任务1准确率: {acc1_replay:.2f}%")
print(f"使用 Replay 算法后，任务2准确率: {acc2_replay:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt



# 2. 制作“任务 1”的数据：只包含数字 0 和 1
idx_task1_train = [i for i, label in enumerate(train_dataset.targets) if label in [0, 1]]
train_loader_task1 = DataLoader(Subset(train_dataset, idx_task1_train), batch_size=64, shuffle=True)
idx_task1_test = [i for i, label in enumerate(test_dataset.targets) if label in [0, 1]]
test_loader_task1 = DataLoader(Subset(test_dataset, idx_task1_test), batch_size=64, shuffle=False)

# 3. 制作“任务 2”的数据：只包含数字 2 和 3
idx_task2_train = [i for i, label in enumerate(train_dataset.targets) if label in [2, 3]]
train_loader_task2 = DataLoader(Subset(train_dataset, idx_task2_train), batch_size=64, shuffle=True)
idx_task2_test = [i for i, label in enumerate(test_dataset.targets) if label in [2, 3]]
test_loader_task2 = DataLoader(Subset(test_dataset, idx_task2_test), batch_size=64, shuffle=False)

print("数据准备完毕！")

# 定义一个非常简单的神经网络
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 4) # 为什么是4？因为我们总共只做0,1,2,3这4个数字的分类
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 训练函数
def train(model, loader, optimizer, criterion, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in loader:
            # 注意：这里的 target 需要映射一下，因为原始标签是0,1,2,3，我们的网络输出是0,1,2,3
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

# 测试准确率的函数
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(loader.dataset)

print("--- 阶段 1：学习任务 1（数字 0 和 1） ---")
train(model, train_loader_task1, optimizer, criterion, epochs=2)
acc1_after_task1 = evaluate(model, test_loader_task1)
print(f"学完任务1，模型对任务1的准确率: {acc1_after_task1:.2f}%\n")

print("--- 阶段 2：直接学习任务 2（数字 2 和 3） ---")
train(model, train_loader_task2, optimizer, criterion, epochs=2)
acc2_after_task2 = evaluate(model, test_loader_task2)
print(f"学完任务2，模型对任务2的准确率: {acc2_after_task2:.2f}%")

# 重点来了：看看它还记不记得旧知识
acc1_after_task2 = evaluate(model, test_loader_task1)
print(f"学完任务2后，回头测试任务1的准确率: {acc1_after_task2:.2f}%  <--- 灾难性遗忘发生了！")

# 重新初始化模型
model_replay = SimpleNN()
optimizer_replay = optim.Adam(model_replay.parameters(), lr=0.0001)

# 创建一个“记忆库”（保留一部分旧数据）
memory_loader = DataLoader(Subset(train_dataset, idx_task1_train[:1000]), batch_size=32, shuffle=True) # 只记住 1000 张旧图

# 1. 正常学任务 1
print("--- Replay 法：学习任务 1 ---")
train(model_replay, train_loader_task1, optimizer_replay, criterion, epochs=2)



# 2. 学任务 2 的时候，穿插着复习记忆库里的旧数据
print("--- Replay 法：学习任务 2（边学边复习） ---")
for epoch in range(2):
    # 先学一点任务 2
    train(model_replay, train_loader_task2, optimizer_replay, criterion, epochs=1)
    # 再复习一点任务 1
    train(model_replay, memory_loader, optimizer_replay, criterion, epochs=1)

# 测试最终结果
acc1_replay = evaluate(model_replay, test_loader_task1)
acc2_replay = evaluate(model_replay, test_loader_task2)
print(f"使用 Replay 算法后，任务1准确率: {acc1_replay:.2f}%")
print(f"使用 Replay 算法后，任务2准确率: {acc2_replay:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from warmup_scheduler import GradualWarmupScheduler

# 1. 准备数据（这里省略具体代码，用你上面能跑通的数据加载代码即可）
# ... (保留你截图里前 20 行准备数据的代码) ...
# 2. 制作“任务 1”的数据：只包含数字 0 和 1
idx_task1_train = [i for i, label in enumerate(train_dataset.targets) if label in [0, 1]]
train_loader_task1 = DataLoader(Subset(train_dataset, idx_task1_train), batch_size=64, shuffle=True)
idx_task1_test = [i for i, label in enumerate(test_dataset.targets) if label in [0, 1]]
test_loader_task1 = DataLoader(Subset(test_dataset, idx_task1_test), batch_size=64, shuffle=False)

# 3. 制作“任务 2”的数据：只包含数字 2 和 3
idx_task2_train = [i for i, label in enumerate(train_dataset.targets) if label in [2, 3]]
train_loader_task2 = DataLoader(Subset(train_dataset, idx_task2_train), batch_size=64, shuffle=True)
idx_task2_test = [i for i, label in enumerate(test_dataset.targets) if label in [2, 3]]
test_loader_task2 = DataLoader(Subset(test_dataset, idx_task2_test), batch_size=64, shuffle=False)
print("数据准备完毕！")

# 2. 定义网络
class SimpleNN__(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        )
    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

# 3. 改造后的 train 函数（重点：加入了调度器）
def train(model, loader, optimizer, criterion, scheduler=None, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
        # 核心修复：每个 epoch 结束，让调度器推进一步（学习率就会慢慢涨）
        if scheduler is not None:
            scheduler.step()

# 4. 测试准确率函数
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(loader.dataset)

# ================= 实验开始 =================

# 5. 初始化模型和优化器
model_replay = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer_replay = optim.Adam(model_replay.parameters(), lr=0.0001)

# 6. 给当前的优化器绑定 Warmup 调度器
# 注意：这里不再手动设置 lr=0.0，让调度器自己接管
scheduler_replay = GradualWarmupScheduler(
    optimizer_replay, 
    multiplier=1.0, 
    total_epoch=3
)
print("模型与调度器就绪！")

# 7. 创建记忆库
memory_loader = DataLoader(Subset(train_dataset, idx_task1_train[:5000]), batch_size=32, shuffle=True)

# --- 开始任务 1 ---
print("--- Replay 法：学习任务 1 ---")
# 传入 scheduler_replay
train(model_replay, train_loader_task1, optimizer_replay, criterion, scheduler=scheduler_replay, epochs=3)
print(f"刚学完任务1，准确率: {evaluate(model_replay, test_loader_task1):.2f}%")

# --- 开始任务 2（同时复习） ---
print("--- Replay 法：学习任务 2（边学边复习） ---")
for epoch in range(3): # 总共跑 3 个 epoch
    # 先学 1 个 epoch 的任务2
    train(model_replay, train_loader_task2, optimizer_replay, criterion, scheduler=scheduler_replay, epochs=1)
    # 再复习 1 个 epoch 的任务1
    train(model_replay, memory_loader, optimizer_replay, criterion, scheduler=scheduler_replay, epochs=1)

# 8. 最终测试
acc1 = evaluate(model_replay, test_loader_task1)
acc2 = evaluate(model_replay, test_loader_task2)
print(f"\n==== 最终成绩 ====")
print(f"使用改进后的Replay + Warmup 算法后，任务1准确率: {acc1:.2f}%")
print(f"使用改进后的Replay + Warmup 算法后，任务2准确率: {acc2:.2f}%")